# 🧪 시나리오 4: 접근 제어 & 보안 검증

## 시나리오
기업 보안팀이 AI Gateway에 대해 다음을 요구합니다:

| # | 보안 요구사항 | APIM 기능 | 기대 결과 |
|---|---|---|---|
| 1 | 인증 없는 요청 차단 | Subscription Key 필수 | 401 |
| 2 | 잘못된 키 차단 | 키 검증 | 401 |
| 3 | 허가된 요청 허용 | 정상 키 | 200 |
| 4 | 존재하지 않는 모델 차단 | 백엔드 라우팅 | 404 |
| 5 | 특정 IP만 허용 | ip-filter 정책 | 403 |

각 셀을 실행하면서 응답 코드와 에러 메시지를 직접 확인하세요.

In [2]:
import os, time, json, subprocess
import requests
from dotenv import load_dotenv

# .env에서 환경 변수 자동 로드
load_dotenv("../../.env")

APIM_URL = os.getenv("APIM_URL")
SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다. .env를 확인하세요."
assert SUBSCRIPTION_KEY, "❌ APIM_SUBSCRIPTION_KEY가 설정되지 않았습니다. .env를 확인하세요."

# APIM 리소스 정보 (테스트 5: 정책 변경용)
APIM_NAME = os.getenv("APIM_NAME")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"
BODY = {"messages": [{"role": "user", "content": "test"}], "max_tokens": 5}

print("✅ 환경 설정 완료")
print(f"   APIM: {APIM_URL}")
print(f"   API: {BASE_URL}")

results = {}  # 테스트 결과 수집용

✅ 환경 설정 완료
   APIM: https://apim-ai-gw-aigateway-20260317.azure-api.net
   API: https://apim-ai-gw-aigateway-20260317.azure-api.net/openai/deployments/gpt-4.1-nano/chat/completions


---
## 테스트 1: 인증 없이 호출

Subscription Key 없이 API를 호출합니다.  
**기대:** `401 Unauthorized`

In [3]:
print("▶ 테스트 1: Subscription Key 없이 호출\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={"Content-Type": "application/json"},  # Key 없음!
    json=BODY,
    timeout=10
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  응답 본문:")
try:
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except Exception:
    print(resp.text[:300])

results["인증 없는 호출 차단"] = resp.status_code == 401
print(f"\n  {'✅ PASS' if resp.status_code == 401 else '❌ FAIL'} - 기대: 401, 실제: {resp.status_code}")

▶ 테스트 1: Subscription Key 없이 호출

  HTTP Status: 401
  응답 본문:
{
  "statusCode": 401,
  "message": "Access denied due to missing subscription key. Make sure to include subscription key when making requests to an API."
}

  ✅ PASS - 기대: 401, 실제: 401


---
## 테스트 2: 잘못된 키로 호출

의도적으로 유효하지 않은 키를 사용합니다.  
**기대:** `401 Unauthorized`

In [4]:
print("▶ 테스트 2: 잘못된 Subscription Key\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": "fake-key-this-should-not-work-12345"
    },
    json=BODY,
    timeout=10
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  응답 본문:")
try:
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except Exception:
    print(resp.text[:300])

results["잘못된 키 차단"] = resp.status_code == 401
print(f"\n  {'✅ PASS' if resp.status_code == 401 else '❌ FAIL'} - 기대: 401, 실제: {resp.status_code}")

▶ 테스트 2: 잘못된 Subscription Key

  HTTP Status: 401
  응답 본문:
{
  "statusCode": 401,
  "message": "Access denied due to invalid subscription key. Make sure to provide a valid key for an active subscription."
}

  ✅ PASS - 기대: 401, 실제: 401


---
## 테스트 3: 정상 키로 호출

올바른 Subscription Key로 호출합니다.  
**기대:** `200 OK`

In [5]:
print("▶ 테스트 3: 정상 Subscription Key\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    },
    json={"messages": [{"role": "user", "content": "Say hello in Korean"}], "max_tokens": 20},
    timeout=30
)

print(f"  HTTP Status: {resp.status_code}")
if resp.status_code == 200:
    data = resp.json()
    print(f"  모델 응답: {data['choices'][0]['message']['content']}")
    print(f"  사용 토큰: {data['usage']}")
else:
    print(f"  응답: {resp.text[:200]}")

results["정상 인증 통과"] = resp.status_code == 200
print(f"\n  {'✅ PASS' if resp.status_code == 200 else '❌ FAIL'} - 기대: 200, 실제: {resp.status_code}")

▶ 테스트 3: 정상 Subscription Key

  HTTP Status: 200
  모델 응답: 안녕하세요 (Annyeonghaseyo)
  사용 토큰: {'completion_tokens': 11, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens': 11, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'total_tokens': 22}

  ✅ PASS - 기대: 200, 실제: 200


---
## 테스트 4: 존재하지 않는 모델 호출 또는 경로에 없는 API 호출

배포되지 않은 모델명으로 호출합니다.  
**기대:** `404 Not Found` 또는 백엔드 에러

In [6]:
print("▶ 테스트 4: 존재하지 않는 모델 호출 (nonexistent-model)\n")

wrong_model_url = f"{APIM_URL}/openai/deployments/nonexistent-model-xyz/chat/completions"

resp = requests.post(
    wrong_model_url,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    },
    json=BODY,
    timeout=30
)

print(f"  HTTP Status: {resp.status_code}")
print(f"  응답 본문:")
try:
    print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
except:
    print(f"  {resp.text[:300]}")

is_blocked = resp.status_code in (404, 400)
results["잘못된 모델 차단"] = is_blocked
print(f"\n  {'✅ PASS' if is_blocked else '❌ FAIL'} - 기대: 404 또는 400, 실제: {resp.status_code}")

if not is_blocked:
    print(f"\n  💡 {resp.status_code}이 반환된 경우:")
    print(f"     백엔드 풀 구성에 따라 다른 에러 코드가 나올 수 있습니다.")
    print(f"     200이 아니면 모델 접근이 차단된 것입니다.")

▶ 테스트 4: 존재하지 않는 모델 호출 또는 경로에 없는 API 호출 (nonexistent-model)

  HTTP Status: 404
  응답 본문:
{
  "error": {
    "type": "invalid_request_error",
    "code": "DeploymentNotFound",
    "message": "The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again."
  }
}

  ✅ PASS - 200이 아니면 정상 차단 (실제: 404)


---
## 테스트 5: IP 필터 정책 (화이트리스트 방식)

존재하지 않는 IP(`1.2.3.4`)만 허용하면, 모든 실제 요청은 **403 Forbidden**으로 차단됩니다.

### 진행 순서

1. Portal → APIM → APIs → Azure OpenAI → All operations → Inbound `</>`
2. `<base />` 바로 아래에 추가 → **Save**:
   ```xml
   <ip-filter action="allow">
       <address>1.2.3.4</address>
   </ip-filter>
   ```
3. **5-A 셀** 실행 → 403 확인
4. Portal에서 `ip-filter` 블록 삭제 → **Save**
5. **5-B 셀** 실행 → 200 확인

In [12]:
print("▶ 테스트 5-A: IP 차단 확인 (화이트리스트에 없으므로 403 기대)\n")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY},
    json=BODY,
    timeout=30
)

print(f"  HTTP Status: {resp.status_code}")
try:
    print(f"  응답: {json.dumps(resp.json(), indent=2, ensure_ascii=False)}")
except:
    print(f"  응답: {resp.text[:300]}")

results["IP 차단 (allow 화이트리스트)"] = resp.status_code == 403
print(f"\n  {'✅ PASS' if resp.status_code == 403 else '❌ FAIL'} - 기대: 403, 실제: {resp.status_code}")

if resp.status_code != 403:
    print(f"\n  💡 403이 아닌 경우:")
    print(f"     1. Portal에서 ip-filter가 정상 저장되었는지 확인")
    print(f"     2. action='allow'이고 address가 192.0.2.1인지 확인")
    print(f"     3. 정책 저장 후 3~5초 대기 필요 (전파 시간)")

▶ 테스트 5-A: IP 차단 확인 (화이트리스트에 없으므로 403 기대)

  HTTP Status: 403
  응답: {
  "statusCode": 403,
  "message": "Forbidden"
}

  ✅ PASS - 기대: 403, 실제: 403


In [14]:
print("▶ 테스트 5-B: 정책 제거 후 정상 호출 확인\n")
print("  📋 Portal에서 ip-filter 블록을 삭제하고 Save 하세요.\n")

input("  ⏸️ 정책을 복원한 후 Enter를 누르세요...")

resp = requests.post(
    BASE_URL,
    params={"api-version": API_VERSION},
    headers={
        "Content-Type": "application/json",
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    },
    json=BODY,
    timeout=30
)

print(f"\n  HTTP Status: {resp.status_code}")
if resp.status_code == 200:
    print(f"  모델 응답: {resp.json()['choices'][0]['message']['content']}")

results["IP 차단 해제 (복원)"] = resp.status_code == 200
print(f"\n  {'✅ PASS' if resp.status_code == 200 else '❌ FAIL'} - 기대: 200, 실제: {resp.status_code}")

if resp.status_code != 200:
    print(f"\n  ⚠️ Portal에서 ip-filter가 완전히 제거되었는지 확인하세요.")

▶ 테스트 5-B: 정책 제거 후 정상 호출 확인

  📋 Portal에서 ip-filter 블록을 삭제하고 Save 하세요.


  HTTP Status: 200
  모델 응답: Hello! How can I

  ✅ PASS - 기대: 200, 실제: 200


---
## 전체 결과 요약

In [15]:
print("═" * 50)
print(" 접근 제어 & 보안 테스트 결과")
print("═" * 50)
print()

total = len(results)
passed = sum(1 for v in results.values() if v)

for check, ok in results.items():
    icon = "✅" if ok else "❌"
    print(f"  {icon} {check}")

print()
print(f"  결과: {passed}/{total} 통과")
print("═" * 50)

══════════════════════════════════════════════════
 접근 제어 & 보안 테스트 결과
══════════════════════════════════════════════════

  ✅ 인증 없는 호출 차단
  ✅ 잘못된 키 차단
  ✅ 정상 인증 통과
  ✅ 잘못된 모델 차단
  ✅ IP 차단 (allow 화이트리스트)
  ✅ IP 차단 해제 (복원)

  결과: 6/6 통과
══════════════════════════════════════════════════
